In [ ]:
import pandas as pd

In [ ]:
# Column names from Freddie Mac documentation
ORIG_COLUMNS = [
    "credit_score", "first_payment_date", "first_time_homebuyer",
    "maturity_date", "msa", "mi_pct", "num_units", "occupancy",
    "cltv", "dti", "original_upb", "ltv", "original_interest_rate",
    "channel", "ppm_flag", "product_type", "property_state",
    "property_type", "postal_code", "loan_sequence_number",
    "loan_purpose", "original_loan_term", "num_borrowers",
    "seller_name", "servicer_name", "super_conforming_flag",
    "pre_harp_loan_seq_num", "program_indicator", "harp_indicator",
    "property_valuation_method", "io_indicator", "mi_cancellation_indicator"
]

In [ ]:
df = pd.read_csv(
    "data/sample_2019/sample_orig_2019.txt",
    sep="|",
    header=None,
    names=ORIG_COLUMNS,low_memory=False
)
df.head()

In [ ]:
df.shape

In [ ]:
df[["credit_score","original_interest_rate", "original_upb", "ltv", "dti", "property_state"]].head(10)

In [ ]:
df[["credit_score","original_interest_rate", "original_upb", "ltv", "dti"]].describe()

In [ ]:
print((df["ltv"] == 999).sum())
print((df["cltv"] == 999).sum())
print((df["mi_pct"] == 999).sum())
print((df["num_borrowers"] == 99).sum())

In [ ]:
import numpy as np

# Replace sentinel value with NaN (missing)
df["credit_score"]=df["credit_score"].replace(9999,np.nan)
df["dti"]=df["dti"].replace(999,np.nan)
df["cltv"]=df["cltv"].replace(999,np.nan)

In [ ]:
df[["credit_score","original_interest_rate", "original_upb", "ltv", "dti","cltv"]].describe()

In [ ]:
# Servicing file column names (From freddi documentation)
SVCG_COLUMNS = [
    "loan_sequence_number", "monthly_reporting_period", "current_actual_upb",
    "current_loan_delinquency_status", "loan_age", "remaining_months_to_maturity",
    "defect_settlement_date", "modification_flag", "zero_balance_code",
    "zero_balance_effective_date", "current_interest_rate", "current_deferred_upb",
    "ddlpi", "mi_recoveries", "net_sales_proceeds", "non_mi_recoveries",
    "total_expenses", "legal_costs", "maintenance_and_preservation_costs",
    "taxes_and_insurance", "miscellaneous_expenses", "actual_loss_calculation",
    "modification_cost", "step_modification_flag", "deferred_payment_plan",
    "estimated_loan_to_value", "zero_balance_removal_upb", "delinquent_accrued_interest",
    "delinquency_due_to_disaster", "borrower_assistance_status_code",
    "current_month_modification_cost", "interest_bearing_upb"
]

svcg=pd.read_csv(
    "data/sample_2019/sample_svcg_2019.txt",
    sep="|",
    header=None,
    names=SVCG_COLUMNS,low_memory=False
)
print(f" Number of rows:{len(svcg):,}")
print(f" Number of columns:{len(svcg.columns)}")


In [ ]:
svcg.head()

In [ ]:
svcg["zero_balance_code"].value_counts(dropna=False)

In [ ]:
prepayments=svcg[svcg["zero_balance_code"]==1.0].copy()
print(f"Number of prepayment events: {len(prepayments):,}")
prepayments[["loan_sequence_number", "monthly_reporting_period", "loan_age", "current_actual_upb"]].head(10)

In [ ]:
# Merge servicing with origination data
combined=pd.merge(svcg,df[["loan_sequence_number", "credit_score", "original_interest_rate", 
        "original_upb", "ltv", "dti", "property_state", "loan_purpose",
        "first_payment_date"]],on="loan_sequence_number",how="left")

print(f"Combined shape: {combined.shape}")
combined.head(10)

In [ ]:
# Average original rate of loans that prepaid vs still alive
prepaid_loans=combined[combined["zero_balance_code"]==1]
still_alive=combined[combined['zero_balance_code'].isna()]
print(f"Avg original rate of PREPAID loans: {prepaid_loans["original_interest_rate"].mean():.3f}%")
print(f"Avg original rate of PREPAID loans: {still_alive["original_interest_rate"].mean():.3f}%")

In [ ]:
import pandas_datareader.data as web
from datetime import datetime

In [ ]:
# Fetch 30 yr US mortgage data from FRED
#start=datetime(2018,1,1)
#end=datetime(2024,12,31)

In [ ]:
#mortgage_rates=web.DataReader("MORTGAGE30US","fred",start,end)
#print(f"Shape:{mortgage_rates.shape}")
#mortgage_rates.head(10)

In [ ]:
# Historical 30-year fixed mortgage rate (monthly avg, %), 2018-2024
# Source: Freddie Mac PMMS via FRED MORTGAGE30US
mortgage_rate_history = {
    "2018-01": 4.03, "2018-02": 4.33, "2018-03": 4.44, "2018-04": 4.47,
    "2018-05": 4.59, "2018-06": 4.57, "2018-07": 4.53, "2018-08": 4.55,
    "2018-09": 4.63, "2018-10": 4.83, "2018-11": 4.87, "2018-12": 4.64,
    "2019-01": 4.46, "2019-02": 4.37, "2019-03": 4.27, "2019-04": 4.14,
    "2019-05": 4.07, "2019-06": 3.80, "2019-07": 3.77, "2019-08": 3.62,
    "2019-09": 3.61, "2019-10": 3.69, "2019-11": 3.70, "2019-12": 3.72,
    "2020-01": 3.62, "2020-02": 3.47, "2020-03": 3.45, "2020-04": 3.31,
    "2020-05": 3.23, "2020-06": 3.16, "2020-07": 3.02, "2020-08": 2.94,
    "2020-09": 2.89, "2020-10": 2.83, "2020-11": 2.77, "2020-12": 2.68,
    "2021-01": 2.74, "2021-02": 2.81, "2021-03": 3.08, "2021-04": 3.06,
    "2021-05": 2.96, "2021-06": 2.98, "2021-07": 2.87, "2021-08": 2.84,
    "2021-09": 2.90, "2021-10": 3.07, "2021-11": 3.07, "2021-12": 3.10,
    "2022-01": 3.45, "2022-02": 3.76, "2022-03": 4.17, "2022-04": 4.98,
    "2022-05": 5.23, "2022-06": 5.52, "2022-07": 5.41, "2022-08": 5.22,
    "2022-09": 6.11, "2022-10": 6.90, "2022-11": 6.81, "2022-12": 6.36,
    "2023-01": 6.27, "2023-02": 6.26, "2023-03": 6.54, "2023-04": 6.34,
    "2023-05": 6.43, "2023-06": 6.71, "2023-07": 6.84, "2023-08": 7.07,
    "2023-09": 7.20, "2023-10": 7.62, "2023-11": 7.44, "2023-12": 6.82,
    "2024-01": 6.64, "2024-02": 6.78, "2024-03": 6.82, "2024-04": 6.99,
    "2024-05": 7.06, "2024-06": 6.92, "2024-07": 6.85, "2024-08": 6.50,
    "2024-09": 6.18, "2024-10": 6.43, "2024-11": 6.81, "2024-12": 6.72
}

mortgage_rates=pd.DataFrame(list(mortgage_rate_history.items()),columns=["year_month","market rate"])
print(f"Shape:{mortgage_rates.shape}")
mortgage_rates.head(10)

In [ ]:
combined["year_month"]=combined['monthly_reporting_period'].astype(str).str[:4]+"-" + combined['monthly_reporting_period'].astype(str).str[4:6]
combined[["monthly_reporting_period","year_month"]].head(10)

In [ ]:
# Merge market rates into combined
combined=pd.merge(combined,mortgage_rates,on="year_month",how='left')

# Verify market_rate is now populated
combined[["loan_sequence_number", "monthly_reporting_period", "year_month", "original_interest_rate", "market rate"]].head(10)

In [ ]:
# rate incentive (note rate - current market rate)
combined['rate_incentive']=combined['original_interest_rate']-combined['market rate']
combined[["loan_sequence_number", "monthly_reporting_period", "year_month", "original_interest_rate", "market rate","rate_incentive"]].head(10)

In [ ]:
combined['rate_incentive'].describe()

In [ ]:
# Building the ML target 

combined['obs_date']=pd.to_datetime(combined["monthly_reporting_period"],format="%Y%m")

# for every loan find the date it prepaid

prepay_dates=combined[combined["zero_balance_code"]==1][["loan_sequence_number","obs_date"]].rename(columns={"obs_date":"prepay_date"})

print(f"Total prepay events: {len(prepay_dates):,}")

In [ ]:
# Add prepay date to combined data

combined=pd.merge(combined,prepay_dates,on="loan_sequence_number",how="left")

combined.head(10)

In [ ]:
combined["months_to_prepay"]=((combined["prepay_date"]-combined["obs_date"]).dt.days/30.44).round()

combined[combined["loan_sequence_number"]=="F19Q10000056"][["obs_date","prepay_date","months_to_prepay"]].head(15)

In [ ]:
# Did the loan prepay in next 12 months

combined["prepay_in_12m"]=((combined["months_to_prepay"]>=1)& (combined["months_to_prepay"]<=12)).astype(int)

combined[combined["loan_sequence_number"]=="F19Q10000056"][["obs_date", "prepay_date", "months_to_prepay", "prepay_in_12m"]].iloc[20:40]

In [ ]:
print(f'total rows:{len(combined):,}')
print(f"Positive labels (prepay in 12m): {combined['prepay_in_12m'].sum():,}")
print(f"Positive rate: {combined['prepay_in_12m'].mean():.2%}")